# 08 — 种子人群 (ABCD) 分布对比

消费 `scripts/compare_seed_distributions.py` 跑出的 `artifacts/<PRODUCT>/seed_compare/` 产物，从 6 个视角看 ABCD 之间 + ABCD vs 建模样本的特征分布差异。

**核心约束 —— 双基准 PSI**：因为 ABCD 只匹配了「最新一天分区」且这一天晚于 `hzz_day.csv` 的 cd_time 跨度，单基准 PSI 会把时间漂移和人群差异混在一起。本 notebook 始终成对呈现 `psi_full`（vs 整个建模 CSV，含时间项）和 `psi_oot`（vs 建模 CSV 最近 15% 切片，时间对齐代理基准）。读图时记住：

- `psi_full` 大、`psi_oot` 小 → 主要是时间漂移，人群本身与总体接近
- `psi_oot` 也大 → 人群本身确实异于建模总体
- 两者都小 → 特征稳定，可放心复用

运行前先跑：
```
PYTHONPATH=src python3 scripts/compare_seed_distributions.py --product hzz_day --seeds-dir data/seeds
```
本 notebook 不会重算 PSI，只画图与切片。

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '../src')
from wdm.analysis.family import parse_families
from wdm.analysis.psi import compute_psi, flag
from wdm.config import load_config
from wdm.preprocess.missing import build_missing_spec, get_spec, to_nan_array
from wdm.utils.binning import equal_freq_edges
from wdm.utils.paths import artifacts_root

PRODUCT = 'hzz_day'
SCOPE = 'selected'   # 'selected' 跑得快；切到 'full' 看全量~970 特征
VERSION = None       # None → 沿用 manifest.json 里记录的 features_version
SEEDS_DIR = '../data/seeds'

cfg = load_config(PRODUCT)
out_root = Path('..').resolve() / artifacts_root(cfg).relative_to(cfg['_repo_root']) / 'seed_compare'
with open(out_root / 'manifest.json', 'r', encoding='utf-8') as f:
    manifest = json.load(f)
if VERSION is None:
    VERSION = manifest['features_version']
scope_dir = out_root / ('selected_{0}'.format(VERSION) if SCOPE == 'selected' else 'full')
print('manifest oot cutoff cd_time :', manifest['cd_time_cutoff'])
print('baseline rows full / oot    :', manifest['baseline_rows_full'], '/', manifest['baseline_rows_oot'])
print('reading from                :', scope_dir)

In [ ]:
psi_matrix = pd.read_csv(scope_dir / 'psi_matrix.csv')
stats_matrix = pd.read_csv(scope_dir / 'stats_matrix.csv')
summary = pd.read_csv(scope_dir / 'summary_per_seed.csv')

SEEDS = list(summary['seed'].astype(str).values)
print('seeds present  :', SEEDS)
print('psi_matrix     :', psi_matrix.shape)
print('stats_matrix   :', stats_matrix.shape)
summary

## 1. 双基准 PSI 散点 —— 拆解「时间」vs「人群」

横轴 `psi_full`、纵轴 `psi_oot`。每个种子一张子图。读法：

- **对角线附近的点**：时间对齐前后 PSI 相近 → 真正的人群差异，无法用「时间漂移」解释掉，需要业务介入。
- **右下偏离对角线**：`psi_full` 大但 `psi_oot` 小 → 主要是建模期 vs 单日的时间漂移，种子人群本身和近期总体接近。
- **左上偏离**：反常情况（`psi_oot` 反而比 `psi_full` 大）；通常说明该特征在最近一段时间内变得更稳定、单日种子反而更接近历史平均；建议复核数据采集口径。

两条参考线：橙线 stable=0.10、红线 broken=0.25。

In [ ]:
ncols = min(len(SEEDS), 4)
nrows = (len(SEEDS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), squeeze=False)
for i, name in enumerate(SEEDS):
    ax = axes[i // ncols][i % ncols]
    x = psi_matrix['{0}_psi_full'.format(name)]
    y = psi_matrix['{0}_psi_oot'.format(name)]
    ax.scatter(x, y, s=8, alpha=0.5)
    lim = float(np.nanmax(np.concatenate([x.values, y.values, [0.3]])))
    ax.plot([0, lim], [0, lim], 'k--', linewidth=0.6, alpha=0.5)
    ax.axhline(0.10, color='orange', linewidth=0.5)
    ax.axhline(0.25, color='red', linewidth=0.5)
    ax.axvline(0.10, color='orange', linewidth=0.5)
    ax.axvline(0.25, color='red', linewidth=0.5)
    ax.set_xlabel('psi_full (vs entire baseline)')
    ax.set_ylabel('psi_oot (vs OOT slice)')
    ax.set_title('Seed {0}  (n={1})'.format(
        name, int(summary.loc[summary['seed'] == name, 'n_rows'].iloc[0])))
    ax.set_xscale('symlog', linthresh=0.01)
    ax.set_yscale('symlog', linthresh=0.01)
for j in range(len(SEEDS), nrows * ncols):
    axes[j // ncols][j % ncols].axis('off')
fig.tight_layout()
plt.show()

## 2. 稳定性 flag 占比 —— 双基准并排

每个种子两根条：左 `psi_full`、右 `psi_oot`。差值大说明时间漂移占主导；两者高度一致说明 PSI 信号就是人群本身。

In [ ]:
fig, ax = plt.subplots(figsize=(max(6, len(SEEDS) * 1.8), 4))
width = 0.35
x = np.arange(len(SEEDS))
colors = {'stable': '#4daf4a', 'shift': '#ff9933', 'broken': '#e41a1c'}
for offset, basis in zip([-width / 2, width / 2], ['full', 'oot']):
    bottom = np.zeros(len(SEEDS))
    for cat in ['stable', 'shift', 'broken']:
        vals = summary['pct_{0}_{1}'.format(cat, basis)].values
        ax.bar(x + offset, vals, width=width, bottom=bottom,
               color=colors[cat], edgecolor='white', linewidth=0.5,
               label=('{0} ({1})'.format(cat, basis)))
        bottom = bottom + vals
ax.set_xticks(x)
ax.set_xticklabels(SEEDS)
ax.set_ylabel('proportion of features')
ax.set_title('Stability flag share — left bar = vs full, right bar = vs oot')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
ax.set_ylim(0, 1.02)
fig.tight_layout()
plt.show()

## 3. Top-N 漂移特征的分布叠加

对每个种子按 `psi_oot` 降序取 top-N，把基准 OOT 切片和 ABCD 5 条线叠在等频直方图上，直观看是「整体偏左/右」还是「缺失变多」。

为避免一次画太多图，默认 `TOP_N = 6`、只画第一个种子；可改 `SEED_FOR_HIST`、`TOP_N` 浏览其他种子。

In [ ]:
SEED_FOR_HIST = SEEDS[0]
TOP_N = 6
BASELINE_PATH = Path('..').resolve() / cfg['data']['train_path']
TIME_COL = cfg['data']['time_column']
OOT_CUTOFF = manifest['cd_time_cutoff']
spec_map = build_missing_spec(cfg)

def _to_nan(series, feat):
    return to_nan_array(series, get_spec(spec_map, feat), analysis=True)[0]

top_path = scope_dir / 'top_shift_{0}.csv'.format(SEED_FOR_HIST)
top_feats = pd.read_csv(top_path).head(TOP_N)['feature'].tolist()
print('top {0} features for seed {1}:'.format(TOP_N, SEED_FOR_HIST), top_feats)

# Load baseline columns lazily (only the columns we need)
base = pd.read_csv(BASELINE_PATH, usecols=[TIME_COL] + top_feats)
oot_mask = base[TIME_COL].astype(np.int64).values >= int(OOT_CUTOFF)
seed_dfs = {n: pd.read_csv(Path(SEEDS_DIR) / '{0}.csv'.format(n),
                            usecols=lambda c, tf=top_feats: c in tf)
            for n in SEEDS}

ncols = 2
nrows = (len(top_feats) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 3.5 * nrows), squeeze=False)
for i, feat in enumerate(top_feats):
    ax = axes[i // ncols][i % ncols]
    arr_oot = _to_nan(base.loc[oot_mask, feat], feat)
    edges = equal_freq_edges(arr_oot, n_bins=10)
    if edges.size < 2:
        ax.set_title('{0}\n(no usable edges)'.format(feat), fontsize=9)
        continue
    arr_oot_clean = arr_oot[~np.isnan(arr_oot)]
    if arr_oot_clean.size:
        ax.hist(np.clip(arr_oot_clean, edges[0], edges[-1] * 0.9999),
                bins=edges, density=True, alpha=0.35,
                color='#888888', label='OOT baseline')
    for name, color in zip(SEEDS, ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']):
        if feat not in seed_dfs[name].columns:
            continue
        arr_s = _to_nan(seed_dfs[name][feat], feat)
        arr_s_clean = arr_s[~np.isnan(arr_s)]
        if arr_s_clean.size == 0:
            continue
        ax.hist(np.clip(arr_s_clean, edges[0], edges[-1] * 0.9999),
                bins=edges, density=True, histtype='step',
                color=color, linewidth=1.4, label=name)
    psi_oot = float(psi_matrix.loc[psi_matrix['feature'] == feat,
                                    '{0}_psi_oot'.format(SEED_FOR_HIST)].iloc[0])
    ax.set_title('{0}  (psi_oot[{1}]={2:.3f})'.format(feat, SEED_FOR_HIST, psi_oot),
                 fontsize=9)
    ax.set_yticks([])
    ax.legend(fontsize=7, loc='best')
for j in range(len(top_feats), nrows * ncols):
    axes[j // ncols][j % ncols].axis('off')
fig.tight_layout()
plt.show()

## 4. 时间窗 family 聚合 —— 信号在哪个业务域上漂移

把特征用 `feature_groups.window_patterns` 聚成 `family_base × window`，对每个种子计算同 family 下的 `mean(psi_oot)`，找出「现金贷信号 / 征信查询 / 多头借贷」哪一块业务域在 ABCD 上漂得最明显。

In [ ]:
fam_df = parse_families(psi_matrix['feature'].tolist(), cfg)
merged = psi_matrix.merge(fam_df, on='feature', how='left')
merged['family_base'] = merged['family_base'].fillna(merged['feature'])

by_family = merged.groupby('family_base').agg(
    **{'n_features': ('feature', 'count'),
       **{('{0}_mean_psi_oot'.format(n)): ('{0}_psi_oot'.format(n), 'mean')
          for n in SEEDS}}
).reset_index()
by_family = by_family[by_family['n_features'] >= 2].sort_values(
    '{0}_mean_psi_oot'.format(SEEDS[0]), ascending=False).head(25)

heat_cols = ['{0}_mean_psi_oot'.format(n) for n in SEEDS]
heat = by_family.set_index('family_base')[heat_cols]
fig, ax = plt.subplots(figsize=(max(5, len(SEEDS) * 1.2), 0.4 * len(heat) + 1.2))
im = ax.imshow(heat.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(SEEDS)))
ax.set_xticklabels(SEEDS)
ax.set_yticks(range(len(heat)))
ax.set_yticklabels(heat.index)
ax.set_title('mean(psi_oot) by family_base × seed  (top 25 families)')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.iloc[i, j]
        if np.isfinite(v):
            ax.text(j, i, '{0:.3f}'.format(v), ha='center', va='center',
                    fontsize=7, color='black' if v < heat.values.max() * 0.5 else 'white')
fig.colorbar(im, ax=ax, fraction=0.025)
fig.tight_layout()
plt.show()

## 5. ABCD 两两交叉 PSI 矩阵 —— 种子彼此像不像

把每个种子当作另一个种子的「期望」，算 PSI(S1 → S2)，对 N 个特征取平均。可以回答：

- A 和 B 是不是同源（都说是高意向，但来自不同数据源）
- C ⊇ D 还是相互独立的两个泛人群

因为 PSI 不对称（用谁的等频边界），矩阵不对称是正常的；关心的是「有没有显著大于 0」。

In [ ]:
MAX_FEATS_FOR_CROSS = 50   # 控制计算量；按 v1_auto 全 200 也能跑
cross_feats = psi_matrix['feature'].head(MAX_FEATS_FOR_CROSS).tolist()
seed_arr = {}
for name in SEEDS:
    sdf = pd.read_csv(Path(SEEDS_DIR) / '{0}.csv'.format(name),
                       usecols=lambda c, cf=cross_feats: c in cf)
    seed_arr[name] = {feat: _to_nan(sdf[feat], feat)
                       for feat in cross_feats if feat in sdf.columns}

mat = np.full((len(SEEDS), len(SEEDS)), np.nan)
for i, a in enumerate(SEEDS):
    for j, b in enumerate(SEEDS):
        if a == b:
            mat[i, j] = 0.0
            continue
        vals = []
        for feat in cross_feats:
            arr_a = seed_arr[a].get(feat)
            arr_b = seed_arr[b].get(feat)
            if arr_a is None or arr_b is None:
                continue
            edges = equal_freq_edges(arr_a, n_bins=10)
            if edges.size < 2:
                continue
            vals.append(compute_psi(arr_a, arr_b, edges=edges, n_bins=10,
                                     missing_as_bin=True))
        mat[i, j] = float(np.mean(vals)) if vals else float('nan')

fig, ax = plt.subplots(figsize=(0.7 * len(SEEDS) + 2, 0.6 * len(SEEDS) + 1.5))
im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=max(0.3, float(np.nanmax(mat))))
ax.set_xticks(range(len(SEEDS)))
ax.set_yticks(range(len(SEEDS)))
ax.set_xticklabels(SEEDS)
ax.set_yticklabels(SEEDS)
ax.set_xlabel('actual (Y)')
ax.set_ylabel('expected (binning source, X)')
ax.set_title('mean PSI(X→Y) over top {0} features'.format(MAX_FEATS_FOR_CROSS))
for i in range(len(SEEDS)):
    for j in range(len(SEEDS)):
        v = mat[i, j]
        if np.isfinite(v):
            ax.text(j, i, '{0:.3f}'.format(v), ha='center', va='center',
                    fontsize=8, color='black')
fig.colorbar(im, ax=ax, fraction=0.04)
fig.tight_layout()
plt.show()

## 6. 缺失率差值 —— 匹配率/数据采集口径问题

`seed_missing_rate - oot_missing_rate`，正值大 → 该特征在种子上比 OOT 更缺失，常见原因：上游表没匹配上、ID 命中率低、跨数据源 schema 不一致。值很负 → 种子上反而非缺失更多（可能是另一种采集口径，少见）。

In [ ]:
miss_long = []
for name in SEEDS:
    s = stats_matrix[['feature', '{0}_missing_rate'.format(name), 'oot_missing_rate']].copy()
    s['delta'] = s['{0}_missing_rate'.format(name)] - s['oot_missing_rate']
    s['seed'] = name
    miss_long.append(s[['feature', 'seed', 'delta']])
miss_long = pd.concat(miss_long, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in SEEDS:
    axes[0].hist(miss_long[miss_long['seed'] == name]['delta'].dropna(),
                 bins=40, histtype='step', label=name, linewidth=1.3)
axes[0].axvline(0, color='k', linewidth=0.5)
axes[0].set_xlabel('seed_missing_rate − oot_missing_rate')
axes[0].set_ylabel('# features')
axes[0].set_title('Distribution of missing-rate delta')
axes[0].legend(fontsize=8)

worst = (miss_long.dropna().sort_values('delta', ascending=False)
         .groupby('seed').head(10))
labels = ['{0} | {1}'.format(r['seed'], r['feature']) for _, r in worst.iterrows()]
axes[1].barh(range(len(worst)), worst['delta'].values, color='#d95f02')
axes[1].set_yticks(range(len(worst)))
axes[1].set_yticklabels(labels, fontsize=7)
axes[1].invert_yaxis()
axes[1].set_xlabel('Δ missing rate')
axes[1].set_title('Top-10 features per seed by Δ missing rate')
fig.tight_layout()
plt.show()

---

### 后续动作建议

1. **大量集中在右下** → 模型迁移到 ABCD 前重训 / 重新校准；时间漂移本身可能由特征工程口径或外部数据源更新触发，建议先排查 cd_time 最近几天的全量分布是否变了。
2. **集中在对角线且 `psi_oot ≥ 0.10`** → 真实的人群差异；新人群上的 lookalike 模型很可能效果衰减，建议构造伪标签或采集少量真实标签做 Stage-2 fine-tune。
3. **某个 family（如 `bureau`）全部跑红** → 上游数据源切换，需对接数据 owner 确认 schema/口径变化。
4. **ABCD 两两交叉 PSI 都接近 0** → ABCD 高度同源，可以合并训练 / 评估；如果 A↔B 接近 0、C↔D 接近 0，但 (A,B) vs (C,D) 显著大 → 4 个种子真分两类，按业务定位分群处理。